# 🧩 GIADA Task 4 — matrice di primitive appaiate
Formula, LUT, interpolazione, Chebyshev, MLP, GRU, physical-τ e direct-z nello stesso contratto held-voltage.

In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_task_4'); GIADA_REPO=WORK/'giada'; TEACHER_REPO=WORK/'neuron_as_deep_net'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip(); print({'revision':REVISION})


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]: del sys.modules[name]
import torch
assert torch.cuda.is_available(),'La Task 4 richiede una GPU CUDA Kaggle.'
from src.giada_teacher import ExtractedGateFormula,PrimitiveMatrixConfig,verified_task3e_root,prepare_primitive_matrix,train_and_freeze_primitive_matrix,evaluate_frozen_primitive_matrix
from src.giada_teacher.primitive_matrix_playground import EXPECTED_TASK3E_ARCHIVE_SHA256,EXPECTED_TASK3E_FINAL_SHA256
prereg=json.loads((GIADA_REPO/'experiments/teacher_primitive_matrix_preregistration_v1.json').read_text())
display({'gpu':torch.cuda.get_device_name(0),'gpu_count':torch.cuda.device_count(),'preregistration':prereg})


In [ ]:
def file_sha(path):
    d=hashlib.sha256()
    with Path(path).open('rb') as h:
        for chunk in iter(lambda:h.read(1024*1024),b''): d.update(chunk)
    return d.hexdigest()
INPUT_ROOT=Path('/kaggle/input'); override=os.environ.get('GIADA_TASK3E_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT_ROOT.is_dir():
    candidates += list(INPUT_ROOT.rglob('giada_joint_m_h_full_repair_confirmation.zip'))
    candidates += list(INPUT_ROOT.rglob('archive.zip'))
    candidates += [p.parent for p in INPUT_ROOT.rglob('final_report.json') if (p.parent/'checkpoint_freeze_report.json').is_file()]
def exact(path):
    try:
        return file_sha(path)==EXPECTED_TASK3E_ARCHIVE_SHA256 if path.is_file() else file_sha(path/'final_report.json')==EXPECTED_TASK3E_FINAL_SHA256
    except Exception: return False
TASK3E_SOURCE=next((p.resolve() for p in candidates if p.exists() and exact(p)),None)
assert TASK3E_SOURCE is not None,'Artefatto esatto giada_joint_m_h_full_repair_confirmation non trovato negli Input Kaggle.'
TASK3E_ROOT=verified_task3e_root(TASK3E_SOURCE,Path('/kaggle/working/.task3e_verified'))
print({'task3e_source':str(TASK3E_SOURCE),'authorization_verified':True})


In [ ]:
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_paired_primitive_matrix')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
config=PrimitiveMatrixConfig(); bundle=prepare_primitive_matrix(formula,config)
display({'families':config.numerical_families+config.learned_families,'seeds':config.seeds,'checkpoints':config.checkpoints,'fit_count':bundle['contract']['fit_count'],'sealed_not_materialized_yet':True})


## 🚀 Training parallelo e freeze
I quattro bracci neurali × tre seed condividono gli stessi minibatch e avanzano nello stesso ciclo GPU. L'output è compatto: una riga per checkpoint.

In [ ]:
training=train_and_freeze_primitive_matrix(bundle,OUTPUT_DIR,config,code_revision=REVISION)
display({'valid':training['valid'],'parallelization':training['parallelization'],'selection':training['selection'],'sealed_accessed':training['sealed_accessed']})
assert training['valid'] and not training['sealed_accessed']


## 🔒 Valutazione sealed unica
La cella seguente materializza il nuovo sealed soltanto dopo il freeze e non modifica checkpoint o modelli.

In [ ]:
final=evaluate_frozen_primitive_matrix(bundle,OUTPUT_DIR,config)
display({'valid':final['valid'],'sealed_contract':final['sealed_contract'],'ranking':final['ranking'],'decision':final['decision']})
assert final['valid'] and not final['selection_used_sealed'] and not final['task3e_sealed_accessed']


## 📦 Download Blob/base64

In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_paired_primitive_matrix','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
